# Phase 3 : IA Générative et Agentique (LLM)
Ce notebook implémente l'analyse de sentiments via des LLMs en utilisant LangChain. Il traite la classification Few-Shot et l'extraction d'aspects (ABSA) avec une sortie structurée via Pydantic, comme demandé dans le cahier des charges de la SAE.

In [7]:
import os
import pandas as pd
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate, FewShotChatMessagePromptTemplate
from pydantic import BaseModel, Field
from typing import List

from src.data_loader import load_yelp_sample

# 1. Chargement d'un avis complexe pour l'expérience
df = load_yelp_sample('../data/raw/review.json', n_rows=100)
sample_review = "The food was absolutely amazing, the steak was cooked to perfection. However, the waiter was very rude and the music was way too loud. The price was okay."

# 2. Initialisation du LLM local (100% Gratuit)
# On appelle DeepSeek-R1 qui tourne sur ta propre machine via Ollama
llm = ChatOllama(model="deepseek-r1", temperature=0)

Chargement de 100 lignes depuis ../data/raw/review.json...


### 1. Classification en Few-Shot
On donne quelques exemples au modèle pour qu'il comprenne le format de réponse attendu pour la classification de sentiment globale.

In [9]:
print("--- 1. Classification en Few-Shot ---")

# Définition des exemples pour le Few-Shot Prompting
examples = [
    {"input": "Hands down the best pizza in Vegas! The crust was perfect.", "output": "POSITIF"},
    {"input": "We waited 45 minutes for a table. The staff was incredibly rude.", "output": "NÉGATIF"},
    {"input": "Bland food and way overpriced. I will not be coming back.", "output": "NÉGATIF"}
]

# Création du template de prompt pour les exemples
example_prompt = ChatPromptTemplate.from_messages([
    ("human", "{input}"),
    ("ai", "{output}"),
])

few_shot_prompt = FewShotChatMessagePromptTemplate(
    example_prompt=example_prompt,
    examples=examples,
)

# Prompt final combinant les instructions système, les exemples et l'avis client
final_prompt = ChatPromptTemplate.from_messages([
    ("system", "Tu es un assistant expert en analyse de sentiment pour des restaurants. Réponds uniquement par POSITIF ou NÉGATIF."),
    few_shot_prompt,
    ("human", "{review}"),
])

# Exécution de la chaîne (Chain) LangChain
chain_few_shot = final_prompt | llm
result_fs = chain_few_shot.invoke({"review": sample_review})

print(f"Avis analysé : {sample_review}")
print(f"Prédiction Few-Shot globale : {result_fs.content}")

--- 1. Classification en Few-Shot ---
Avis analysé : The food was absolutely amazing, the steak was cooked to perfection. However, the waiter was very rude and the music was way too loud. The price was okay.
Prédiction Few-Shot globale : NÉGATIF


### 2. Extraction d'aspects structurée (ABSA) avec LangChain
C'est la partie Agentique avancée : on force le LLM à répondre avec une structure JSON stricte pour qu'elle soit exploitable par le reste du code Python.

In [6]:
print("\n--- 2. Aspect-Based Sentiment Analysis (Sortie Structurée) ---")

# 1. Définition du schéma de données attendu via Pydantic
class AspectSentiment(BaseModel):
    aspect: str = Field(description="L'aspect mentionné dans l'avis (ex: nourriture, service, ambiance, prix)")
    sentiment: str = Field(description="Le sentiment associé à cet aspect ('POSITIF' ou 'NÉGATIF')")

class ReviewAnalysis(BaseModel):
    aspects: List[AspectSentiment] = Field(description="Liste des aspects extraits de l'avis et leurs sentiments")

# 2. On contraint le LLM à respecter cette structure (API Tools d'OpenAI via LangChain)
structured_llm = llm.with_structured_output(ReviewAnalysis)

# 3. Création du Prompt métier
absa_prompt = ChatPromptTemplate.from_messages([
    ("system", "Tu es un analyste de données spécialisé dans l'hôtellerie-restauration. Extrais tous les aspects mentionnés dans l'avis suivant et attribue un sentiment (POSITIF ou NÉGATIF) à chacun."),
    ("human", "{review}")
])

# 4. Exécution
chain_absa = absa_prompt | structured_llm
result_absa = chain_absa.invoke({"review": sample_review})

# 5. Affichage propre du résultat JSON / Pydantic
print("Résultat de l'extraction :")
for item in result_absa.aspects:
    print(f" - Aspect : {item.aspect.upper()} | Sentiment : {item.sentiment}")


--- 2. Aspect-Based Sentiment Analysis (Sortie Structurée) ---
Résultat de l'extraction :
 - Aspect : NOURRITURE | Sentiment : POSITIF
 - Aspect : STEAK | Sentiment : POSITIF
 - Aspect : SERVEUR | Sentiment : NÉGATIF
 - Aspect : MUSIQUE | Sentiment : NÉGATIF
 - Aspect : PRIX | Sentiment : NÉGATI
